In [18]:
import pytesseract
from PIL import Image
import pandas as pd
import os
import glob
import string
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Point pytesseract to the local Tesseract installation
pytesseract.pytesseract.tesseract_cmd = r'E:\Tesseract-OCR\tesseract.exe'

In [19]:
# Load image names and their correct (ground truth) text
df = pd.read_csv('Image name and annotation.csv')
df.head()

,Image name,Ground truth label
0,1_test,"When people ask what I see in you, I just smil..."
1,2_test,The quick brown fox jumped over the 5 lazy dogs!
2,3_test,Creating an OCR Communication App with Tessera...
3,4_test,Long-Term Care Insurance
4,5_test,How to print a line break in Python


Text normalization function

In [20]:
def normalize_text(text):
    """Clean and standardize text before comparison."""
    text = text.lower()
    text = text.replace('\n', ' ')
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join(text.split())
    return text

Preprocessing functions

In [21]:
def preprocess_image_global(image_path):
    """Convert image to black/white using a single global threshold."""
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresholded = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return thresholded


def preprocess_image_adaptive(image_path):
    """Convert image to black/white using adaptive (local) thresholding."""
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    adaptive = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=25,
        C=15
    )
    return adaptive

Baseline run (no preprocessing)

In [22]:
# Run OCR on all images WITHOUT any preprocessing (baseline)
baseline_results = []

for index, row in df.iterrows():
    image_name = row['Image name']
    ground_truth = row['Ground truth label']

    matching_files = glob.glob(f'Test_images/{image_name}.*')
    if len(matching_files) == 0:
        print(f"Warning: no image found for {image_name}")
        continue

    image_path = matching_files[0]
    image = Image.open(image_path)
    extracted_text = pytesseract.image_to_string(image)

    clean_extracted = normalize_text(extracted_text)
    clean_ground_truth = normalize_text(ground_truth)
    is_match = (clean_extracted == clean_ground_truth)

    baseline_results.append({
        'image_name': image_name,
        'ground_truth': ground_truth,
        'extracted_text': extracted_text,
        'is_match': is_match
    })

baseline_df = pd.DataFrame(baseline_results)
baseline_accuracy = (baseline_df['is_match'].sum() / len(baseline_df)) * 100
print(f"Baseline accuracy (no preprocessing): {baseline_accuracy:.1f}%")

Baseline accuracy (no preprocessing): 60.0%


Preprocessing applied to ALL images (for comparison)

In [ ]:
# Run OCR on all images WITH adaptive preprocessing applied to everything
all_preprocessed_results = []

for index, row in df.iterrows():
    image_name = row['Image name']
    ground_truth = row['Ground truth label']

    matching_files = glob.glob(f'Test_images/{image_name}.*')
    if len(matching_files) == 0:
        continue

    image_path = matching_files[0]
    processed = preprocess_image_adaptive(image_path)
    extracted_text = pytesseract.image_to_string(processed)

    clean_extracted = normalize_text(extracted_text)
    clean_ground_truth = normalize_text(ground_truth)
    is_match = (clean_extracted == clean_ground_truth)

    all_preprocessed_results.append({
        'image_name': image_name,
        'is_match': is_match
    })

all_preprocessed_df = pd.DataFrame(all_preprocessed_results)
all_preprocessed_accuracy = (all_preprocessed_df['is_match'].sum() / len(all_preprocessed_df)) * 100
print(f"Accuracy with ADAPTIVE preprocessing applied to ALL images: {all_preprocessed_accuracy:.1f}%")

Accuracy with preprocessing applied to ALL images: 36.7%


In [26]:
# Run OCR on all images WITH global (OTSU) preprocessing, for comparison
global_preprocessed_results = []

for index, row in df.iterrows():
    image_name = row['Image name']
    ground_truth = row['Ground truth label']

    matching_files = glob.glob(f'Test_images/{image_name}.*')
    if len(matching_files) == 0:
        continue

    image_path = matching_files[0]
    processed = preprocess_image_global(image_path)
    extracted_text = pytesseract.image_to_string(processed)

    clean_extracted = normalize_text(extracted_text)
    clean_ground_truth = normalize_text(ground_truth)
    is_match = (clean_extracted == clean_ground_truth)

    global_preprocessed_results.append({
        'image_name': image_name,
        'is_match': is_match
    })

global_preprocessed_df = pd.DataFrame(global_preprocessed_results)
global_preprocessed_accuracy = (global_preprocessed_df['is_match'].sum() / len(global_preprocessed_df)) * 100
print(f"Accuracy with GLOBAL (OTSU) preprocessing on all images: {global_preprocessed_accuracy:.1f}%")

Accuracy with GLOBAL (OTSU) preprocessing on all images: 46.7%


Final approach — Selective preprocessing (best result)

In [24]:
# Selective strategy:
# 1. Try OCR without preprocessing first.
# 2. Only apply preprocessing if the first attempt fails.
final_results = []

for index, row in df.iterrows():
    image_name = row['Image name']
    ground_truth = row['Ground truth label']

    matching_files = glob.glob(f'Test_images/{image_name}.*')
    if len(matching_files) == 0:
        print(f"Warning: no image found for {image_name}")
        continue

    image_path = matching_files[0]
    clean_ground_truth = normalize_text(ground_truth)

    # Attempt 1: original image, no preprocessing
    image = Image.open(image_path)
    text_original = pytesseract.image_to_string(image)
    clean_original = normalize_text(text_original)

    if clean_original == clean_ground_truth:
        final_results.append({
            'image_name': image_name,
            'ground_truth': ground_truth,
            'final_text': text_original,
            'method_used': 'original',
            'is_match': True
        })
    else:
        # Attempt 2: apply adaptive preprocessing
        processed = preprocess_image_adaptive(image_path)
        text_processed = pytesseract.image_to_string(processed)
        clean_processed = normalize_text(text_processed)
        is_match_after = (clean_processed == clean_ground_truth)

        final_results.append({
            'image_name': image_name,
            'ground_truth': ground_truth,
            'final_text': text_processed,
            'method_used': 'preprocessed',
            'is_match': is_match_after
        })

final_df = pd.DataFrame(final_results)
final_accuracy = (final_df['is_match'].sum() / len(final_df)) * 100
print(f"Final accuracy (selective preprocessing): {final_accuracy:.1f}%")

Final accuracy (selective preprocessing): 63.3%


In [27]:

print("SUMMARY OF RESULTS")
print(f"Baseline (no preprocessing):           {baseline_accuracy:.1f}%")
print(f"Global (OTSU) preprocessing - all:     {global_preprocessed_accuracy:.1f}%")
print(f"Adaptive preprocessing - all images:   {all_preprocessed_accuracy:.1f}%")
print(f"Selective preprocessing (final):       {final_accuracy:.1f}%")

SUMMARY OF RESULTS
Baseline (no preprocessing):           60.0%
Global (OTSU) preprocessing - all:     46.7%
Adaptive preprocessing - all images:   36.7%
Selective preprocessing (final):       63.3%
